# C-SPAN Archives API — Advanced

Every optional parameter on every endpoint, plus query recipes and
cross-endpoint workflows. Start with [`notebook.ipynb`](notebook.ipynb) if you
haven't used the client before.

**Sections**

1. [Client configuration](#1-client-configuration)
2. [`/mentions` — all filters](#2-mentions--all-filters)
3. [`/bills` — pagination and analysis](#3-bills--pagination-and-analysis)
4. [`/people` — search strategies](#4-people--search-strategies)
5. [`/people/{personId}` — resolving and batching](#5-peoplepersonid--resolving-and-batching)
6. [`/programs/search` — Lucene recipes](#6-programssearch--lucene-recipes)
7. [`/programs/{videoId}` — enrichment](#7-programsvideoid--enrichment)
8. [Cross-endpoint workflows](#8-cross-endpoint-workflows)
9. [Exporting datasets](#9-exporting-datasets)
10. [Errors, retries, and validation](#10-errors-retries-and-validation)

## 1. Client configuration

Beyond the API key, the client takes a timeout, a retry policy, a default
output format, and an optional pre-built `requests.Session`.

In [ ]:
import os
from getpass import getpass

from cspan import CSpanClient

if not os.environ.get("CSPAN_API_KEY"):
    os.environ["CSPAN_API_KEY"] = getpass("C-SPAN API key: ")

client = CSpanClient(
    timeout=60.0,          # seconds per request (default 30)
    max_retries=5,         # retries for 429 / 5xx (default 3)
    backoff_factor=1.0,    # exponential backoff base (default 0.5)
    output_format="json",  # default for every call
)
client

In [ ]:
# Use it as a context manager to close the HTTP session when you're done.
with CSpanClient(output_format="records") as c:
    rows = c.bills("appropriations")
    print(len(rows), "rows, already a list[dict] because output_format='records'")

## 2. `/mentions` — all filters

Every documented parameter: `query` (required), `limit`, `cursor`, `personid`,
`date`, `maxdate`, `mindate`, `page`, `videotype`.

In [ ]:
# Every filter at once: a phrase, spoken by one person, in a date window,
# restricted to one video type, paged.
result = client.mentions(
    "climate",
    personid=6153,          # Nancy Pelosi (from /people)
    mindate="2020-01-01",
    maxdate="2021-01-01",
    videotype="Speech",
    limit=5,
    page=1,
)

for row in result["mentions"]:
    print(row["begintime"][:10], "|", row["programTitle"][:50])
    print("   ", row["text"][:90], "...")

### Exact phrases vs. loose terms

Wrap a phrase in quotes to match it as a unit. Unquoted terms match more
loosely and return far more (noisier) results.

In [ ]:
loose = client.mentions("inflation reduction act", limit=1)
exact = client.mentions('"Inflation Reduction Act"', limit=1)

print("loose first hit:", loose["mentions"][0]["text"][:110])
print()
print("exact first hit:", exact["mentions"][0]["text"][:110])

### `date` vs. `mindate` / `maxdate`

`date` pins a single day; `mindate` / `maxdate` define an inclusive window. All
three take `yyyy-mm-dd` and are validated client-side before the request.

In [ ]:
single_day = client.mentions("climate", date="2024-02-17", limit=3)
window = client.mentions("climate", mindate="2024-02-01", maxdate="2024-02-29", limit=3)

print("single day:", [r["begintime"][:10] for r in single_day["mentions"]])
print("window:    ", [r["begintime"][:10] for r in window["mentions"]])

### Manual cursor control

`iter_records` handles pagination for you, but you can drive the cursor
yourself when you need to checkpoint progress or stop on a condition.

In [ ]:
cursor = None
seen = 0

for page_num in range(3):
    page = client.mentions("climate", limit=5, cursor=cursor)
    rows = page["mentions"]
    seen += len(rows)
    cursor = page.get("cursor")
    print(f"page {page_num + 1}: {len(rows)} rows, next cursor: {str(cursor)[:24]}...")
    if not rows or not cursor:
        break

print("total:", seen)

### Comparing how often a phrase is spoken, by year

In [ ]:
phrase = '"artificial intelligence"'
counts = {}

for year in range(2019, 2025):
    rows = list(
        client.iter_records(
            client.mentions,
            query=phrase,
            mindate=f"{year}-01-01",
            maxdate=f"{year}-12-31",
            max_items=60,           # cap so the demo stays quick and quota-friendly
        )
    )
    counts[year] = len(rows)

for year, n in counts.items():
    print(f"{year}  {'#' * (n // 5):<40} {n}")

## 3. `/bills` — pagination and analysis

Parameters: `query` (required) and `cursor`. The response carries `billnumber`,
`billtitle`, `congress`, and `id`.

In [ ]:
import collections

rows = list(client.iter_records(client.bills, query="climate", max_items=300))
by_congress = collections.Counter(r["congress"] for r in rows)

print(f"{len(rows)} bills mentioning 'climate'\n")
for congress, n in sorted(by_congress.items(), reverse=True)[:10]:
    print(f"  {congress}th Congress: {n}")

In [ ]:
# Straight into pandas for analysis.
df = client.bills("healthcare", format="dataframe")
df[["congress", "billnumber", "billtitle"]].head(10)

## 4. `/people` — search strategies

Parameters: `query`, `first`, `last`, `cursor` — **all optional**. `query`
matches names *and* titles, which makes it useful for role-based searches.

In [ ]:
# Three different ways to search.
by_last = client.people(last="Pelosi", format="records")
by_full = client.people(first="Nancy", last="Pelosi", format="records")
by_title = client.people(query="Senator", format="records")

print("last='Pelosi':          ", len(by_last), "results")
print("first+last:             ", len(by_full), "results")
print("query='Senator' (title):", len(by_title), "results")
print()
for row in by_title[:5]:
    print(f"  {row['name']:<25} {row['title']}")

In [ ]:
# Build a name -> id lookup you can reuse for personid filters elsewhere.
senators = {}
for row in client.iter_records(client.people, query="Senator", max_items=100):
    senators[row["name"]] = row["id"]

print(len(senators), "people with 'Senator' in their name or title")
list(senators.items())[:5]

## 5. `/people/{personId}` — resolving and batching

Takes the numeric `id` from `/people`. Note the response shape: a single person
still comes back wrapped in a `people` list.

**Caveat worth knowing:** an unknown or non-numeric person ID returns HTTP 200
with an empty payload — *not* a 404. Check for the `people` key rather than
catching `NotFoundError`.

In [ ]:
from cspan import to_records

def resolve_person(name):
    """Search by name, then fetch the full record for the first match."""
    hits = client.people(query=name, format="records")
    if not hits:
        return None
    return to_records(client.person(hits[0]["id"]))[0]

for name in ["Nancy Pelosi", "Chuck Schumer", "Mitch McConnell"]:
    person = resolve_person(name)
    if person:
        print(f"{person['id']:>8}  {person['name']:<20}  {person['title']}")

In [ ]:
# An unknown ID is an empty 200, not a 404 — handle it by checking the payload.
missing = client.person(999999999)
print("keys:", list(missing.keys()))
print("found:", bool(missing.get("people")))

## 6. `/programs/search` — Lucene recipes

Parameters: `query` (required), `cursor`, `sort`.

Valid query fields: `abstract`, `category`, `date`, `format`, `isbn`,
`location`, `person`, `personid`, `series`, `sponsor`, `subject`, `tag`, `text`.

`sort` accepts `date` or `popular`, and **must** include a direction — the
client raises `ValidationError` if you forget it.

In [ ]:
queries = {
    "free text":        "climate",
    "single field":     "text:climate",
    "by person id":     "personid:6153",
    "by category":      "category:Senate",
    "boolean AND":      "text:climate AND category:Senate",
    "boolean OR":       "text:(climate OR emissions)",
    "exclusion":        "text:climate NOT category:Senate",
    "exact phrase":     'text:"climate change"',
    "wildcard":         "text:climat*",
    "grouped fields":   "category:Senate AND text:(climate OR emissions)",
}

for label, q in queries.items():
    try:
        hits = client.programs_search(q, format="records")
        top = hits[0]["title"][:45] if hits else "(no results)"
        print(f"{label:<16} {len(hits):>3} hits   {top}")
    except Exception as exc:
        print(f"{label:<16} error: {type(exc).__name__}: {exc}")

**Note on unsupported syntax.** Not every Lucene construct is wired up on the
server. Range queries such as `date:[2024-01-01 TO 2024-12-31]` return
**HTTP 500**, surfaced as an `APIError`. To filter programs by date, sort with
`sort="date desc"` and filter client-side, or use `/mentions`, which has real
`mindate` / `maxdate` parameters.

In [ ]:
from cspan import APIError

# Server-side range queries are not supported:
try:
    client.programs_search("text:climate AND date:[2024-01-01 TO 2024-12-31]")
except APIError as exc:
    print("APIError:", exc.status_code)

# Do this instead — sort, then filter locally.
rows = client.programs_search("text:climate", sort="date desc", format="records")
in_2024 = [r for r in rows if r["date"].startswith("2024")]
print(f"{len(in_2024)} of {len(rows)} returned programs are from 2024")

In [ ]:
# Sorting: newest first vs. most popular. Direction is mandatory.
newest = client.programs_search("text:climate", sort="date desc", format="records")
popular = client.programs_search("text:climate", sort="popular desc", format="records")

print("newest first:")
for row in newest[:3]:
    print("  ", row["date"][:10], row["title"][:55])

print("\nmost popular:")
for row in popular[:3]:
    print("  ", row["date"][:10], row["title"][:55])

In [ ]:
from cspan import ValidationError

# A sort without a direction is rejected before the request is sent.
try:
    client.programs_search("text:climate", sort="date")
except ValidationError as exc:
    print("ValidationError:", exc)

## 7. `/programs/{videoId}` — enrichment

Takes the numeric `id` from `/programs/search`. Unlike the other endpoints, a
single program comes back as a **flat dict**, not wrapped in a list.

**Caveat:** despite what the API docs suggest, a public ID string (e.g.
`"556839-1"`) returns **HTTP 400**, surfaced as an `APIError`. Use the numeric
`id`. An unknown numeric id correctly returns 404 / `NotFoundError`.

In [ ]:
from cspan import APIError, NotFoundError

# Numeric id works:
program = client.program(683870)
print("OK:", program["title"], "| publicId:", program["publicId"])

# Public ID string does not:
try:
    client.program("556839-1")
except APIError as exc:
    print("APIError:", exc.status_code, "-", str(exc)[:60])

# Unknown numeric id -> 404:
try:
    client.program(999999999)
except NotFoundError as exc:
    print("NotFoundError:", exc.status_code)

In [ ]:
# Enrich search results with the detail each program only exposes individually
# (duration, publicId, video link).
hits = client.programs_search("text:climate", sort="date desc", format="records")

for row in hits[:5]:
    detail = client.program(row["id"])
    minutes = detail.get("videoDuration", 0) / 60
    print(f"{detail['date'][:10]}  {minutes:>6.1f} min  {detail['title'][:50]}")

## 8. Cross-endpoint workflows

The API has no structured joins, so real research means chaining calls. These
three recipes cover the common shapes.

### 8a. Mentions → programs

A mention tells you *what was said*; the program record tells you the *context*
it was said in. Join them through `programPublicId`.

In [ ]:
mentions = client.mentions('"climate change"', limit=5, format="records")

for m in mentions:
    print(f"{m['begintime'][:10]}  {m['person'][:35]}")
    print(f"    program: {m['programTitle'][:55]} ({m['programPublicId']})")
    print(f"    said:    {m['text'][:80]}...")

### 8b. People → programs

Resolve a person once, then use their numeric id in both `/mentions`
(`personid=`) and `/programs/search` (`personid:` Lucene field).

In [ ]:
hits = client.people(first="Nancy", last="Pelosi", format="records")
pid = hits[0]["id"]
print(f"Nancy Pelosi = id {pid}\n")

spoke = client.mentions("healthcare", personid=pid, limit=3, format="records")
appeared = client.programs_search(f"personid:{pid}", sort="date desc", format="records")

print(f"mentions of 'healthcare' by her: {len(spoke)}")
for m in spoke:
    print("   ", m["begintime"][:10], m["text"][:70], "...")

print(f"\nrecent programs featuring her: {len(appeared)}")
for p in appeared[:3]:
    print("   ", p["date"][:10], p["title"][:60])

### 8c. `speeches_on_bill` — the built-in research helper

The API has no bill→speech link, so this helper searches the spoken transcript
for the bill's title and number (several phrasings), keeps floor video types,
and groups the results by senator.

It makes many requests, so cap it with `max_items` while exploring.

In [ ]:
result = client.speeches_on_bill(
    title="Inflation Reduction Act",
    number="H.R. 5376",
    congress=117,
    videotypes=("Speech", "Debate"),
    max_items=30,   # keep the demo short; raise or drop for real runs
)

print("bill:      ", result["bill"])
print("videotypes:", result["videotypes"])
print("senators:  ", len(result["senators"]))
print()
for s in result["senators"][:10]:
    print(f"  {len(s['speeches']):>3} speeches  {s['name']} ({s['title']})")

## 9. Exporting datasets

`save()` combines fetch + paginate + write. It accepts a directory (auto-named
`<endpoint>.<ext>`) or an explicit file path.

| Format | Requires |
| --- | --- |
| `csv`, `json`, `jsonl` | nothing extra |
| `xlsx` | pandas + openpyxl |
| `parquet` | pandas + pyarrow |

In [ ]:
# Full result set (follows the cursor) into a directory, auto-named.
print(client.save(client.bills, "out/", query="budget", format="csv", max_items=200))

# Explicit path, first page only, pretty JSON.
print(client.save("mentions", "out/ai.json", query='"artificial intelligence"',
                  format="json", paginate=False))

# JSON Lines — a good fit for streaming large exports.
print(client.save(client.people, "out/", query="Senator", format="jsonl", max_items=200))

In [ ]:
# Excel / Parquet need pandas plus an engine.
try:
    print(client.save(client.bills, "out/", query="climate",
                      format="xlsx", max_items=100))
except ImportError as exc:
    print("install the extra first:", exc)

In [ ]:
# Standalone converters, if you already have a payload in hand.
from cspan import to_csv, to_dataframe, to_jsonl, to_records

payload = client.people(last="Pelosi")

print("records:", len(to_records(payload)))
print("csv head:", to_csv(payload)[:80])
print("jsonl head:", to_jsonl(payload)[:80])
to_dataframe(payload).head(3)

## 10. Errors, retries, and validation

The exception hierarchy:

```
CSpanError                  # base — catch-all
├── ValidationError         # bad input, before the request (also a ValueError)
└── APIError                # API returned an error (.status_code, .response)
    ├── AuthenticationError # 401/403 — missing/invalid key
    ├── NotFoundError       # 404
    └── RateLimitError      # 429 — has .retry_after
```

In [ ]:
from cspan import ValidationError

# Everything below fails locally — no request is sent.
bad_inputs = [
    ("empty query",        lambda: client.bills("")),
    ("bad date",           lambda: client.mentions("ai", date="2024-13-01")),
    ("date wrong format",  lambda: client.mentions("ai", mindate="01/01/2024")),
    ("limit zero",         lambda: client.mentions("ai", limit=0)),
    ("negative page",      lambda: client.mentions("ai", page=-1)),
    ("sort no direction",  lambda: client.programs_search("climate", sort="date")),
    ("unknown format",     lambda: client.bills("budget", format="yaml")),
    ("missing person id",  lambda: client.person("")),
]

for label, call in bad_inputs:
    try:
        call()
        print(f"{label:<20} (no error)")
    except ValidationError as exc:
        print(f"{label:<20} {exc}")

In [ ]:
from cspan import APIError, CSpanError, RateLimitError

# A production-shaped handler: most specific first, CSpanError as the net.
try:
    data = client.mentions("climate", limit=5)
except RateLimitError as exc:
    print("rate limited; retry after", exc.retry_after, "seconds")
except APIError as exc:
    print("API error", exc.status_code, exc)
except CSpanError as exc:
    print("client error:", exc)
else:
    print("got", len(data["mentions"]), "rows")

In [ ]:
# Transient failures (429, 5xx) are retried automatically with exponential
# backoff, honoring Retry-After. Tune the policy per client.
patient = CSpanClient(max_retries=8, backoff_factor=2.0, timeout=120.0)
print("retries:", 8, "| backoff base:", 2.0, "| timeout:", patient.timeout)
patient.close()

### Watch your quota

The API enforces a quota that deep pagination will hit — a few hundred requests
in quick succession is enough to start getting 429s, and `iter_records` issues
one request per page. Practical habits:

- Always pass `max_items` while exploring; drop it only for a real harvest.
- Prefer one wide crawl saved to disk over re-running the same query.
- For long jobs, raise `max_retries` and `backoff_factor` as above.
- Catch `RateLimitError` and honor `.retry_after` if you're running unattended.

In [ ]:
import time

from cspan import RateLimitError

def harvest(endpoint, *, retries=3, **params):
    """Paginate an endpoint, pausing and resuming if the quota is hit."""
    rows = []
    for attempt in range(retries):
        try:
            rows = list(client.iter_records(endpoint, **params))
            break
        except RateLimitError as exc:
            wait = exc.retry_after or 60 * (attempt + 1)
            print(f"quota hit; sleeping {wait:.0f}s")
            time.sleep(wait)
    return rows

print(len(harvest(client.bills, query="transportation", max_items=40)), "rows")

### Empty results

When a search matches nothing the API omits the result key and returns only a
cursor. The client normalizes that to an empty result set, so `records` is
`[]`, `csv` is `""`, and `iter_records` simply yields nothing — you never get a
phantom row containing the cursor.

In [ ]:
nothing = client.mentions("zzzqqxnotarealphrase")

print("raw json: ", nothing)
print("records:  ", client.mentions("zzzqqxnotarealphrase", format="records"))
print("csv:      ", repr(client.mentions("zzzqqxnotarealphrase", format="csv")))
print("iterated: ", list(client.iter_records(client.mentions,
                                             query="zzzqqxnotarealphrase")))

### A note on CSV exports

The "Enable Search CSV Exports" toggle in the C-SPAN developer portal affects
the portal's own web UI, not the REST API — the API returns JSON on every
endpoint regardless. All CSV, Excel, and Parquet output in this client is
produced locally by `format=` and `save()`.